# ASGC remote worker (Colab)

This notebook turns one Colab runtime into one **ASGC worker** that trains against a remote
parameter server over the public internet. Open it in several Colab sessions (different Google
accounts, different runtimes: T4, CPU, TPU-host CPU, and so on). Each session is one worker in a real
heterogeneous asynchronous cluster.

What each worker does on every step:
1. Computes a gradient on its own shard of the training set.
2. Compresses it with the codec the **server** chose (Top-K / QSGD / identity), with error feedback.
3. Pushes it to the server, gets a `PushAck` (accepted / downweighted / rejected for staleness).
4. Pulls the latest weights back as an INT8 delta.

**Before you run:**
- Use the same `TASK` as the server.
- Give every worker a different `RANK` (0, 1, 2, ...) and the same `NUM_SHARDS` (the number of workers).
- Store the server secret as a Colab secret named `ASGC_AUTHKEY` (key icon in the left sidebar,
  enable notebook access). Otherwise the notebook asks for it.

In [ ]:
# ---- Worker configuration -------------------------------------------------
SERVER = "tcp://129.225.103.140:29500"   # parameter server address
TASK = "fmnist"                           # must match the server: fmnist | cifar10
RANK = 0                                  # unique per worker: 0, 1, 2, ...
NUM_SHARDS = 3                            # total number of workers
WORKER_ID = f"colab-r{RANK}"              # shows up in the server's logs and events.jsonl
BATCH_SIZE = None                         # None = task default (128)
LOCAL_STEPS = 1                           # >1 averages that many micro-batches per push
USE_AMP = True                            # fp16 autocast on GPU runtimes
REPO_URL = "https://github.com/ksacheth/parallel-computing.git"
REPO_BRANCH = "main"

In [ ]:
import getpass
import os

try:
    from google.colab import userdata
    AUTHKEY = userdata.get("ASGC_AUTHKEY")
except Exception:
    AUTHKEY = None
if not AUTHKEY:
    AUTHKEY = getpass.getpass("ASGC_AUTHKEY: ")
os.environ["ASGC_AUTHKEY"] = AUTHKEY.strip()
print("authkey loaded (" + str(len(os.environ["ASGC_AUTHKEY"])) + " chars)")

In [ ]:
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path("/content/parallel-computing")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## Connectivity check

Opens a plain TCP connection to the server and reports the round-trip time. If this fails, the
server is down or the port is not open (the OCI Security List and the VM firewall both have to allow it).

In [ ]:
import socket
import time
from urllib.parse import urlparse

parsed = urlparse(SERVER)
times = []
for _ in range(5):
    t0 = time.perf_counter()
    with socket.create_connection((parsed.hostname, parsed.port), timeout=10):
        times.append((time.perf_counter() - t0) * 1000)
print(f"TCP connect to {parsed.hostname}:{parsed.port} OK, median RTT {sorted(times)[2]:.1f} ms")

## Train

Runs until the server stops the run (`--max-versions` / `--max-wall-time-s` on the server), or until
you interrupt this cell. The server logs every push to `results/distributed/<run>/events.jsonl`.

In [ ]:
cmd = [
    sys.executable, "-u", "scripts/ps_worker.py",
    "--server", SERVER,
    "--task", TASK,
    "--rank", str(RANK),
    "--num-shards", str(NUM_SHARDS),
    "--worker-id", WORKER_ID,
    "--local-steps", str(LOCAL_STEPS),
    "--data-root", "/content/data",
]
if BATCH_SIZE:
    cmd += ["--batch-size", str(BATCH_SIZE)]
if USE_AMP:
    cmd += ["--amp"]

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    if proc.poll() is None:
        proc.terminate()   # interrupting the cell stops the worker cleanly
    proc.wait()
print("worker exited with code", proc.returncode)